# 03 特征筛选模块 (core.selectors)

基于 `hscredit.xlsx` 真实信贷数据，演示多种特征筛选策略的实际效果。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (init_setting,
    VarianceSelector, NullSelector, ModeSelector, CorrSelector,
    VIFSelector, IVSelector, LiftSelector, PSISelector,
    TypeSelector, RegexSelector, FeatureImportanceSelector,
    Chi2Selector, FTestSelector, MutualInfoSelector,
    CompositeFeatureSelector, SelectionReportCollector,
)
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
target = 'FPD15'
features = [c for c in df.columns if c not in ['MOB1','MOB2','loan_date','FPD15','SFPD15']]
X = df[features].fillna(-999); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'原始特征数: {len(features)}, 样本数: {len(df):,}, 坏率: {y.mean():.2%}')

## 1. 缺失率筛选

In [ ]:
null_sel = NullSelector(threshold=0.5)
null_sel.fit(X_tr, y_tr)
print(f'缺失率>50%剔除后: {null_sel.transform(X_tr).shape[1]} 个特征')
# 查看各特征缺失率
missing = X_tr.isnull().mean().sort_values(ascending=False)
print('缺失率最高的特征:')
print(missing.head(10))

## 2. 方差筛选

In [ ]:
var_sel = VarianceSelector(threshold=0.0)
var_sel.fit(X_tr, y_tr)
print(f'方差筛选后: {var_sel.transform(X_tr).shape[1]} 个特征')

## 3. IV 筛选

In [ ]:
iv_sel = IVSelector(threshold=0.02, max_n_bins=8)
iv_sel.fit(X_tr, y_tr)
X_iv = iv_sel.transform(X_tr)
print(f'IV>0.02筛选后: {X_iv.shape[1]} 个特征')
print('保留特征:', X_iv.columns.tolist())

## 4. 相关性筛选

In [ ]:
corr_sel = CorrSelector(threshold=0.85)
corr_sel.fit(X_iv, y_tr)
X_corr = corr_sel.transform(X_iv)
print(f'相关性筛选后: {X_corr.shape[1]} 个特征')
print('保留:', X_corr.columns.tolist())

## 5. PSI 稳定性筛选

In [ ]:
psi_sel = PSISelector(threshold=0.25, max_n_bins=8)
psi_sel.fit(X_tr, y_tr, X_test=X_te)
X_psi = psi_sel.transform(X_tr)
print(f'PSI<0.25筛选后: {X_psi.shape[1]} 个特征')

## 6. 特征重要性筛选

In [ ]:
fi_sel = FeatureImportanceSelector(threshold=0.005, n_estimators=100, random_state=42)
fi_sel.fit(X_corr, y_tr)
X_fi = fi_sel.transform(X_corr)
print(f'重要性筛选后: {X_fi.shape[1]} 个特征')
print('保留:', X_fi.columns.tolist())

## 7. 正则表达式筛选（按特征命名规律）

In [ ]:
# 只保留12个月维度的特征
regex_sel = RegexSelector(pattern='_12m$')
regex_sel.fit(X_tr)
print('12m特征:', regex_sel.transform(X_tr).columns.tolist())

# 逾期相关特征
regex_overdue = RegexSelector(pattern='overdue')
regex_overdue.fit(X_tr)
print('逾期特征:', regex_overdue.transform(X_tr).columns.tolist())

## 8. CompositeFeatureSelector — 完整筛选流水线

In [ ]:
collector = SelectionReportCollector()
comp_sel = CompositeFeatureSelector([
    ('null',  NullSelector(threshold=0.5)),
    ('var',   VarianceSelector(threshold=0.0)),
    ('iv',    IVSelector(threshold=0.02, max_n_bins=8)),
    ('corr',  CorrSelector(threshold=0.85)),
    ('psi',   PSISelector(threshold=0.25, max_n_bins=8)),
], report_collector=collector)
comp_sel.fit(X_tr, y_tr, **{'psi__X_test': X_te})
X_final = comp_sel.transform(X_tr)
print(f'原始: {X_tr.shape[1]} → 筛选后: {X_final.shape[1]} 个特征')
print('最终特征:', X_final.columns.tolist())
print('\n筛选报告:')
collector.report()